# Flux Emergence (Theory) — Cheung & Isobe (2014)
## Implementation / 구현

This notebook reproduces key theoretical concepts from Cheung & Isobe (2014), Living Rev. Solar Phys., 11, 3.

이 노트북은 Cheung & Isobe (2014) 리뷰의 주요 이론 개념을 수치적으로 재현한다.

**Topics / 주제**
1. Thin flux tube rise with buoyancy and drag / 부력과 항력을 포함한 가는 자속관 상승
2. Magnetic buoyancy vs pressure scale height condition / 자기 부력 대 압력 척도 높이 조건
3. Kink instability criterion $\Phi > 2\pi$ / 꼬임 불안정성 기준
4. Emergence of an arched flux loop (2D schematic) / 호형 자속 루프 출현 개요도
5. Parker (undular) instability growth rate vs wavelength / 파커(물결) 불안정성 성장률
6. Sunspot bipole schematic after emergence / 출현 후 흑점 쌍극 개요도


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# Physical constants (CGS)
G_SOLAR = 2.74e4   # cm/s^2, solar surface gravity
MM = 1e8           # cm per Mm
DAY = 86400.0      # s per day

## 1. Thin Flux Tube Rise / 가는 자속관 상승

We integrate a 1D equation of motion for a buoyant flux tube rising through a polytropic stratification from the base of the convection zone.

등폴리트로픽 층화에서 대류층 바닥으로부터 부력으로 상승하는 1차원 자속관 운동방정식을 적분한다.

$$\frac{dy}{dt} = v,\qquad \frac{dv}{dt} = g\,\frac{\Delta\rho}{\rho} - \frac{C_D\,\rho_{\text{amb}}(y)}{\pi a \rho_{\text{amb}}(y)}\,v|v|$$

With $B \propto \rho^{1/2}$ scaling (horizontal pancake limit near surface) and $\Delta\rho/\rho = -1/\beta(y)$.

In [ ]:
def rise_equations(state, t, params):
    """ODE system for a buoyant thin flux tube rising through stratification.

    Args:
        state: Tuple (y, v) where y is height above base [cm], v is rise velocity [cm/s].
        t: Time [s] (unused for autonomous system).
        params: Dict with keys 'rho0', 'p0', 'B0', 'Hp', 'a_tube', 'CD'.

    Returns:
        Tuple (dy/dt, dv/dt) derivatives.
    """
    y, v = state
    rho_amb = params['rho0'] * np.exp(-y / params['Hp'])
    p_amb = params['p0'] * np.exp(-y / params['Hp'])
    B = params['B0'] * (rho_amb / params['rho0']) ** 0.5
    beta = 8.0 * np.pi * p_amb / B ** 2
    drho_over_rho = -1.0 / beta
    a_buoy = G_SOLAR * drho_over_rho
    a_drag = -params['CD'] * v * abs(v) / (np.pi * params['a_tube'])
    return [v, a_buoy + a_drag]

params = {
    'rho0': 1e-1,
    'p0': 1e13,
    'B0': 2e4,
    'Hp': 15 * MM,
    'a_tube': 1 * MM,
    'CD': 1.0,
}

t = np.linspace(0, 10 * DAY, 2000)
sol = odeint(rise_equations, [0.0, 0.0], t, args=(params,))
y_Mm = sol[:, 0] / MM
v_kms = sol[:, 1] / 1e5

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(t / DAY, y_Mm, 'b-')
axes[0].axhline(20, color='r', ls='--', label='surface (20 Mm)')
axes[0].set_xlabel('Time [days] / 시간')
axes[0].set_ylabel('Height above base [Mm] / 고도')
axes[0].set_title('Thin flux tube rise / 자속관 상승')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(t / DAY, v_kms, 'g-')
axes[1].set_xlabel('Time [days] / 시간')
axes[1].set_ylabel('Rise velocity [km/s] / 상승 속도')
axes[1].set_title('Velocity evolution / 속도 진화')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Magnetic Buoyancy vs Pressure Scale Height / 자기 부력 대 압력 척도 높이

The pressure scale height $H_p = -(d\ln p/dz)^{-1}$ sets how rapidly a rising flux tube must expand. Near the photosphere $H_p \sim 150$ km; deep in the CZ it exceeds 50 Mm.

압력 척도 높이 $H_p$는 상승 자속관의 팽창 속도를 결정한다. 광구 근처 ~150 km, 대류층 깊은 곳은 50 Mm 초과.

In [ ]:
def pressure_scale_height(z_Mm):
    """Approximate pressure scale height in the solar convection zone.

    Args:
        z_Mm: Depth below photosphere in Mm (positive = deeper).

    Returns:
        Scale height in Mm using a piecewise fit matching Spruit (1974) CZ model.
    """
    return 0.15 + 0.5 * z_Mm ** 0.9

z = np.linspace(0, 200, 500)
Hp = pressure_scale_height(z)

B = 1e4
rho_photo = 3e-7
rho = rho_photo * np.exp(z / 30.0)
p = rho * (1e6 + 2e5 * z)
beta = 8 * np.pi * p / B ** 2
buoy_accel = G_SOLAR / beta

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(z, Hp, 'b-', lw=2, label='$H_p$ [Mm]')
ax1.set_xlabel('Depth below photosphere [Mm] / 광구 하부 깊이')
ax1.set_ylabel('Pressure scale height [Mm]', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.semilogy(z, buoy_accel, 'r-', lw=2, label='buoyancy acc.')
ax2.set_ylabel('Buoyancy acceleration [cm/s$^2$]', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title('$H_p$ and buoyancy vs depth / 척도 높이·부력 가속도 대 깊이')
plt.tight_layout()
plt.show()

print(f'Hp at photosphere: {pressure_scale_height(0)*1000:.1f} km')
print(f'Hp at 20 Mm depth: {pressure_scale_height(20):.2f} Mm')
print(f'Hp at base CZ (200 Mm): {pressure_scale_height(200):.2f} Mm')

## 3. Kink Instability Criterion / 꼬임 불안정성 기준

For a twisted flux tube with profile $B_l(r) = B_0 \exp(-r^2/a^2)$, $B_\theta(r) = q\,r\,B_l(r)$, Linton et al. (1996) derived the critical twist:

$$q_{\rm cr} = a^{-1},\qquad \Phi = q L = \text{total twist over length } L$$

Instability requires $\Phi > 2\pi$ (more than one full turn over an Alfvén length).

Linton et al.(1996)은 임계 꼬임 $q_{\rm cr}=1/a$를 유도했다. 불안정 조건은 총 꼬임 $\Phi>2\pi$ (Alfvén 길이당 1회전 초과).

In [ ]:
def kink_stable(q, a_tube):
    """Return True if twist parameter q is below critical value 1/a_tube.

    Args:
        q: Twist parameter [1/length].
        a_tube: Flux tube radius [same length unit as 1/q].

    Returns:
        Boolean stability flag.
    """
    return q < 1.0 / a_tube

a_tube = 1.0
q_values = np.linspace(0, 2.0, 500)
stability = np.array([kink_stable(q, a_tube) for q in q_values])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(q_values, 0, 1, where=stability, color='lightgreen', alpha=0.5, label='Stable')
ax.fill_between(q_values, 0, 1, where=~stability, color='lightcoral', alpha=0.5, label='Kink unstable')
ax.axvline(1.0 / a_tube, color='k', ls='--', label='$q_{cr}=1/a$')
ax.set_xlabel('Twist parameter $q$ [1/a] / 꼬임 파라미터')
ax.set_ylabel('Stability indicator')
ax.set_ylim(0, 1)
ax.set_title('Kink instability criterion $q > 1/a$ / 꼬임 불안정성 기준')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Critical q for a_tube=1: q_cr = {1.0/a_tube:.2f}')
print('Equivalent total twist threshold Phi > 2*pi (one full turn per Alfven length).')

## 4. Arched Flux Loop Emergence (2D schematic) / 호형 자속 루프 출현 개요도

Schematic 2D potential-field approximation of an emerged bipolar arcade.

출현한 쌍극 아케이드의 정전 자기장 근사 2D 개요도.

In [ ]:
def dipole_field(x, z, x0, z0, m):
    """2D magnetic dipole field components.

    Args:
        x, z: Spatial grids [arbitrary units].
        x0, z0: Dipole center coordinates.
        m: Dipole moment magnitude and sign.

    Returns:
        Tuple (Bx, Bz) of field components.
    """
    dx, dz = x - x0, z - z0
    r2 = dx ** 2 + dz ** 2 + 1e-6
    Bx = m * (2 * dx * dz) / r2 ** 2
    Bz = m * (dz ** 2 - dx ** 2) / r2 ** 2
    return Bx, Bz

x = np.linspace(-20, 20, 300)
z = np.linspace(-2, 15, 220)
X, Z = np.meshgrid(x, z)

Bx1, Bz1 = dipole_field(X, Z, -5, -2, +1.0)
Bx2, Bz2 = dipole_field(X, Z, +5, -2, -1.0)
Bx = Bx1 + Bx2
Bz = Bz1 + Bz2

fig, ax = plt.subplots(figsize=(9, 5))
speed = np.sqrt(Bx ** 2 + Bz ** 2)
ax.streamplot(X, Z, Bx, Bz, color=np.log10(speed + 1e-3), cmap='plasma', density=2.0, linewidth=0.8)
ax.axhline(0, color='k', lw=2)
ax.plot(-5, 0, 'ro', ms=15, label='N polarity')
ax.plot(+5, 0, 'bo', ms=15, label='S polarity')
ax.set_xlabel('x [Mm]')
ax.set_ylabel('z [Mm] (photosphere at z=0)')
ax.set_title('Emerged arched flux loop / 출현한 호형 자속 루프')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 5. Parker (Undular) Instability Growth Rate / 파커(물결) 불안정성 성장률

For a horizontal flux sheet in isothermal atmosphere, the undular mode has linear dispersion (dimensionless):

$$\omega^2 \approx -\frac{g}{H_p}\!\left(1 - \frac{1}{2\beta}\right) + k^2 v_A^2$$

Instability ($\omega^2<0$) occurs for $k < k_{\rm cr} = \sqrt{g/(H_p v_A^2)\cdot(1-1/(2\beta))}$.

등온 대기의 수평 자속판에서 물결 모드는 위와 같은 분산 관계를 따른다. $k<k_{\rm cr}$에서 불안정.

In [ ]:
def undular_growth_rate(k, g_over_Hp, vA2, beta):
    """Compute undular-mode growth rate (positive when unstable).

    Args:
        k: Wavenumber array [1/length].
        g_over_Hp: g/Hp in [1/time^2].
        vA2: Alfven speed squared [length^2/time^2].
        beta: Plasma beta (dimensionless).

    Returns:
        Array of growth rates [1/time]; zero where modes are stable.
    """
    omega_sq = -g_over_Hp * (1.0 - 1.0 / (2.0 * beta)) + k ** 2 * vA2
    gamma = np.where(omega_sq < 0, np.sqrt(-omega_sq), 0.0)
    return gamma

Hp = 1.0
g = 1.0
for beta in [1.0, 2.0, 5.0, 20.0]:
    vA2 = 2.0 / beta
    k_arr = np.linspace(0.01, 3.0, 400) / Hp
    gamma = undular_growth_rate(k_arr, g / Hp, vA2, beta)
    plt.plot(k_arr * Hp, gamma, label=f'beta = {beta}')

plt.xlabel('k H$_p$ / 무차원 파수')
plt.ylabel('Growth rate gamma [(g/Hp)^(1/2)]')
plt.title('Undular (Parker) instability growth rate / 물결 불안정성 성장률')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Sunspot Bipole Schematic / 흑점 쌍극 개요도

Idealized photospheric vertical magnetic field $B_z(x,y)$ of a mature bipolar active region, modeled as two Gaussian flux concentrations with opposite polarity.

성숙한 쌍극 활동 영역의 이상화된 광구 수직 자기장 $B_z(x,y)$를 두 개의 가우시안 자속 집중(반대 극성)으로 모형화한다.

In [ ]:
def gaussian_spot(X, Y, x0, y0, sigma, amp):
    """Gaussian vertical-field profile for a sunspot.

    Args:
        X, Y: 2D coordinate grids [Mm].
        x0, y0: Spot center [Mm].
        sigma: Gaussian width [Mm].
        amp: Peak field strength [G] (positive or negative).

    Returns:
        2D array B_z [G].
    """
    return amp * np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2 * sigma ** 2))

x = np.linspace(-25, 25, 400)
y = np.linspace(-15, 15, 240)
X, Y = np.meshgrid(x, y)

Bz = gaussian_spot(X, Y, -8, 0, 3, +3000) + gaussian_spot(X, Y, +8, 0, 3, -3000)

np.random.seed(0)
for _ in range(40):
    cx, cy = np.random.uniform(-20, 20), np.random.uniform(-10, 10)
    sign = 1 if cx < 0 else -1
    Bz += gaussian_spot(X, Y, cx, cy, 0.8, sign * 300 * np.random.rand())

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(Bz, extent=(-25, 25, -15, 15), origin='lower', cmap='RdBu_r', vmin=-2500, vmax=2500)
plt.colorbar(im, ax=ax, label='B_z [G]')
ax.contour(X, Y, Bz, levels=[0], colors='k', linewidths=1.0)
ax.set_xlabel('x [Mm]')
ax.set_ylabel('y [Mm]')
ax.set_title('Bipolar AR magnetogram schematic / 쌍극 AR 자기도 개요')
plt.tight_layout()
plt.show()

total_flux_pos = np.sum(Bz[Bz > 0]) * (x[1] - x[0]) * (y[1] - y[0]) * 1e16
print(f'Total positive unsigned flux: {total_flux_pos:.2e} Mx')

## Summary / 요약

We reproduced six key theoretical elements of flux emergence from Cheung & Isobe (2014):

1. **Thin-tube rise** — a buoyant tube from 20 Mm depth reaches the surface in a few days, consistent with $B \propto \rho^{1/2}$ scaling.
2. **Scale height stratification** — $H_p$ grows from 150 km at the surface to ~100 Mm deep in the CZ, setting the horizontal pancake behavior.
3. **Kink instability** — twisted tubes with $q > 1/a$ (equivalently $\Phi > 2\pi$) writhe and can produce delta spots.
4. **Bipolar arcade** — emerged $\Omega$-loop produces canonical current-free arched field after reaching force-free relaxation.
5. **Parker instability** — growth rate peaks at intermediate k, decreasing as $\beta$ increases.
6. **AR magnetogram** — two-spot Gaussian + plage scattering produces realistic bipolar morphology with polarity inversion line.

Cheung & Isobe (2014) 리뷰의 여섯 가지 핵심 이론 요소를 재현했다: 얇은 관 상승, 척도 높이 층화, 꼬임 불안정, 쌍극 아케이드, Parker 불안정 성장률, AR 자기도.